In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, transform, data
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,Dense,Dropout,MaxPooling2D,Flatten,BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from keras.utils import np_utils
from keras.losses import SparseCategoricalCrossentropy
from keras import callbacks
from keras.models import load_model
from sklearn import model_selection
import pickle
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.model_selection import train_test_split
import seaborn as sns
from tensorflow.keras.applications import ResNet50, vgg16

## Preprocess

In [ ]:
train_dir = "../input/brain-tumor-classification-mri/Training"
test_dir = "../input/brain-tumor-classification-mri/Testing"
classes = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']

def show_train_history(train_history,train,validation):
    plt.plot(train_history.history[train])
    plt.plot(train_history.history[validation])
    plt.title('Train History')
    plt.ylabel(train)
    plt.xlabel('Epoch')
    plt.legend(['train','validation'],loc='upper left')
    plt.show()

def roundoff(arr):
    """To round off according to the argmax of each predicted label array. """
    arr[np.argwhere(arr != arr.max())] = 0
    arr[np.argwhere(arr == arr.max())] = 1
    return arr

# Image sample 1
img=image.load_img(train_dir+'/'+'no_tumor'+'/'+'5.jpg')
print((image.img_to_array(img)).shape)
plt.imshow(img)
plt.show()
img=image.load_img(train_dir+'/'+'no_tumor'+'/'+'5.jpg',target_size=(100,100))
print((image.img_to_array(img)).shape)
plt.imshow(img)
plt.show()

# Image sample 2
img=image.load_img(train_dir+'/'+'glioma_tumor'+'/'+'gg (10).jpg')
print((image.img_to_array(img)).shape)
plt.imshow(img)
plt.show()
img=image.load_img(train_dir+'/'+'glioma_tumor'+'/'+'gg (10).jpg',target_size=(100,100))
print((image.img_to_array(img)).shape)
plt.imshow(img)
plt.show()

In [ ]:
# Read train data into x and y arrays
x=[]
y=[]
for folder in classes:
    image_list=os.listdir(train_dir+"/"+folder)
    for img_name in image_list:
        # Loading images
        img=image.load_img(train_dir+"/"+folder+"/"+img_name,target_size=(100,100))

        # Converting to arrays
        img=image.img_to_array(img)

        x.append(img) # appending image array
        y.append(classes.index(folder)) # appending class index to the array
print("Preparing Training Dataset Completed.")

# Read test data into x and y arrays
x_test=[]
y_test=[]
for folder in classes:
    image_list=os.listdir(test_dir+"/"+folder)
    for img_name in image_list:
        # Loading images
        img=image.load_img(test_dir+"/"+folder+"/"+img_name,target_size=(100,100))

        # Converting to arrays
        img=image.img_to_array(img)

        x_test.append(img) # appending image array
        y_test.append(classes.index(folder)) # appending class index to the array
print("Preparing Testing Dataset Completed.")

# Prepare train and validation datasets
x = np.array(x)
y = to_categorical(y, 4)
print(y.shape)

x_train, x_val, y_train, y_val = train_test_split(x,y,test_size=0.25,random_state=5)

x_test = np.array(x_test)
y_test = to_categorical(y_test, 4)

#x_train=x_train.reshape(x_train.shape[0],x_train.shape[1],x_train.shape[2],1)
#x_test=x_test.reshape(x_test.shape[0],x_test.shape[1],x_test.shape[2],1)
#x_val=x_val.reshape(x_val.shape[0],x_val.shape[1],x_val.shape[2],1)

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print('x_val shape:', x_val.shape)
print(x_val.shape[0], 'validation samples')
print('x_test shape:', x_test.shape)
print(x_test.shape[0], 'test samples')
print(y_test.shape)

## Train with Optimization

### SGD

In [ ]:
model=ResNet50(input_shape=(100,100,3),weights=None,classes=4)
model.compile(optimizer='SGD',loss='categorical_crossentropy',metrics=['accuracy'])
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
history = model.fit(x=x_train,y=y_train,epochs=35,batch_size=32,validation_data=(x_val,y_val))
#pickle.dump(history.history,open(global_path+'/resnet-history.his','wb'))
#保存模型
#model.save(global_path+'/resnet.h5')
show_train_history(history,'accuracy','val_accuracy')

In [ ]:
# Test
scores = model.evaluate(x_test,y_test)
print(scores[1])

# Evalute
prediction = model.predict(x_test)
for labels in prediction:
    labels = roundoff(labels)

# Confusion matrix
classes = np.array(classes)
print(metrics.classification_report(y_test, prediction, target_names=classes))
print(metrics.roc_auc_score(y_test, prediction))

In [ ]:
# Test
scores = model.evaluate(x_val,y_val)
print(scores[1])

# Evalute
prediction = model.predict(x_val)
for labels in prediction:
    labels = roundoff(labels)

# Confusion matrix
classes = np.array(classes)
print(metrics.classification_report(y_val, prediction, target_names=classes))
print(metrics.roc_auc_score(y_val, prediction))

### Genetic Algorithm

In [ ]:
class Genetic:
    
    def __init__(self,pop_size,nlayers,max_nfilters,max_sfilters,model):
        self.pop_size = pop_size
        self.nlayers = nlayers
        self.max_nfilters = max_nfilters
        self.max_sfilters = max_sfilters
        self.max_acc = 0
        self.best_arch = np.zeros((1,6))
        self.gen_acc = []
        self.model = model
    
    def generate_population(self):
        np.random.seed(0)
        pop_nlayers = np.random.randint(1,self.max_nfilters,(self.pop_size,self.nlayers))
        pop_sfilters = np.random.randint(1,self.max_sfilters,(self.pop_size,self.nlayers))
        pop_total = np.concatenate((pop_nlayers,pop_sfilters),axis=1)
        return pop_total
    
    def select_parents(self,pop,nparents,fitness):
        parents = np.zeros((nparents,pop.shape[1]))
        for i in range(nparents):
            best = np.argmax(fitness)
            parents[i] = pop[best]
            fitness[best] = -99999
        return parents
    
    def crossover(self,parents):
        nchild = self.pop_size - parents.shape[0]
        nparents = parents.shape[0]
        child = np.zeros((nchild,parents.shape[1]))
        for i in range(nchild):
            first = i % nparents
            second = (i+1) % nparents
            child[i,:2] = parents[first][:2]
            child[i,2] = parents[second][2]
            child[i,3:5] = parents[first][3:5]
            child[i,5] = parents[second][5]
        return child

    def mutation(self,child):
        for i in range(child.shape[0]):
            val = np.random.randint(1,6)
            ind = np.random.randint(1,4) - 1
            if child[i][ind] + val > 100:
                child[i][ind] -= val
            else:
                child[i][ind] += val
            val = np.random.randint(1,4)
            ind = np.random.randint(4,7) - 1
            if child[i][ind] + val > 20:
                child[i][ind] -= val
            else:
                child[i][ind] += val
        return child
    
    def fitness(self,pop,epochs):
        pop_acc = []
        for i in range(pop.shape[0]):
            nfilters = pop[i][0:3]
            sfilters = pop[i][3:]
            (self.model).compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
            H = (self.model).fit(x=x_train,y=y_train,epochs=epochs,batch_size=32,validation_data=(x_val,y_val))
            acc = H.history['accuracy']
            pop_acc.append(max(acc)*100)
        if max(pop_acc) > self.max_acc:
            self.max_acc = max(pop_acc)
            self.best_arch = pop[np.argmax(pop_acc)]
        self.gen_acc.append(max(pop_acc))
        return pop_acc
    
    def smooth_curve(self,factor,gen):
        smoothed_points = []
        for point in self.gen_acc:
            if smoothed_points:
                prev = smoothed_points[-1]
                smoothed_points.append(prev*factor + point * (1-factor))
            else:
                smoothed_points.append(point)
        plt.plot(range(gen+1),smoothed_points,'g',label='Smoothed training acc')
        plt.xticks(np.arange(gen+1))
        plt.legend()
        plt.title('Fitness Accuracy vs Generations')
        plt.xlabel('Generations')
        plt.ylabel('Fitness (%)')
        plt.show()
    
    def evaluation_val(self):
        # Test
        _scores = (self.model).evaluate(x_val,y_val)
        print(_scores[1])
        
        # Evaluate
        _prediction = (self.model).predict(x_val)
        for labels in _prediction:
            labels = roundoff(labels)
        
        # Confusion matrix
        _classes = np.array(['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor'])
        print(metrics.classification_report(y_val, _prediction, target_names=_classes))
        print("AUC score: ", metrics.roc_auc_score(y_val, _prediction))
    
    def evaluation_test(self):
        # Test
        _scores = (self.model).evaluate(x_test,y_test)
        print(_scores[1])
        
        # Evaluate
        _prediction = (self.model).predict(x_test)
        for labels in _prediction:
            labels = roundoff(labels)
        
        # Confusion matrix
        _classes = np.array(['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor'])
        print(metrics.classification_report(y_test, _prediction, target_names=_classes))
        print("AUC score: ", metrics.roc_auc_score(y_test, _prediction))

In [ ]:
pop_size = 5
nlayers = 3
max_nfilters = 100
max_sfilters = 20
epochs = 20
num_generations = 2

model = ResNet50(input_shape=(100,100,3),weights=None,classes=4)
genCNN = Genetic(pop_size,nlayers,max_nfilters,max_sfilters,model)
pop = genCNN.generate_population()

for i in range(num_generations+1):
    pop_acc = genCNN.fitness(pop,epochs)
    print('Best Accuracy at the generation {}: {}'.format(i,genCNN.max_acc))
    parents = genCNN.select_parents(pop,5,pop_acc.copy())
    child = genCNN.crossover(parents)
    child = genCNN.mutation(child)
    pop = np.concatenate((parents,child),axis=0).astype('int')

In [ ]:
#genCNN.fitness(pop,epochs)
genCNN.evaluation_val()
genCNN.evaluation_test()

### Adam

In [ ]:
model=ResNet50(input_shape=(100,100,3),weights=None,classes=4)
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
history = model.fit(x=x_train,y=y_train,epochs=35,batch_size=32,validation_data=(x_val,y_val))
#pickle.dump(history.history,open(global_path+'/resnet-history.his','wb'))
#保存模型
#model.save(global_path+'/resnet.h5')
show_train_history(history,'accuracy','val_accuracy')

In [ ]:
# Test
scores = model.evaluate(x_test,y_test)
print(scores[1])

# Evalute
prediction = model.predict(x_test)
for labels in prediction:
    labels = roundoff(labels)

# Confusion matrix
classes = np.array(classes)
print(metrics.classification_report(y_test, prediction, target_names=classes))
print("AUC score: ", metrics.roc_auc_score(y_test, prediction))

In [ ]:
# Test
scores = model.evaluate(x_val,y_val)
print(scores[1])

# Evalute
prediction = model.predict(x_val)
for labels in prediction:
    labels = roundoff(labels)

# Confusion matrix
classes = np.array(classes)
print(metrics.classification_report(y_val, prediction, target_names=classes))
print("AUC score: ", metrics.roc_auc_score(y_val, prediction))